# Low-intensity STED fluorescence microscopy denoising

This example shows how to denoise low-intensity STED fluorescence microscopy
images of live-cell mitochondria using the pretrained foundation model
[`deepinv.models.RAM`](https://deepinv.org/api/stubs/deepinv.models.RAM.html). We load real Abberior STED microscopy data
from (Osuna-Vargas et al., 2025), process it in batches, and
visualize the results both with [`deepinv.utils.plot`](https://deepinv.org/api/stubs/deepinv.utils.plot.html) and with the
interactive 3D viewer [`deepinv.utils.plot_napari`](https://deepinv.org/api/stubs/deepinv.utils.plot_napari.html).

Acquiring fluorescence microscopy images at low excitation laser power reduces
photobleaching and phototoxicity, which lets biologists image living samples for
longer, but it produces noisier images. Here we recover clean images from
acquisitions taken at 1.5 microwatt excitation, using images taken at 8 microwatt
as the ground truth.

This example requires `tifffile`, `rarfile` and `napari`. Install them with
``pip install tifffile rarfile "napari[all]"``.

<!-- MathJax macro definitions inserted automatically -->
$$
\newcommand{\forw}[1]{{A\left({#1}\right)}}
\newcommand{\noise}[1]{{N\left({#1}\right)}}
\newcommand{\inverse}[1]{{R\left({#1}\right)}}
\newcommand{\inversef}[2]{{R\left({#1},{#2}\right)}}
\newcommand{\inversename}{R}
\newcommand{\reg}[1]{{g_\sigma\left({#1}\right)}}
\newcommand{\regname}{g_\sigma}
\newcommand{\sensor}[1]{{\eta\left({#1}\right)}}
\newcommand{\datafid}[2]{{f\left({#1},{#2}\right)}}
\newcommand{\datafidname}{f}
\newcommand{\distance}[2]{{d\left({#1},{#2}\right)}}
\newcommand{\distancename}{d}
\newcommand{\denoiser}[2]{{\operatorname{D}_{{#2}}\left({#1}\right)}}
\newcommand{\denoisername}{\operatorname{D}_{\sigma}}
\newcommand{\xset}{\mathcal{X}}
\newcommand{\yset}{\mathcal{Y}}
\newcommand{\group}{\mathcal{G}}
\newcommand{\metric}[2]{{d\left({#1},{#2}\right)}}
\newcommand{\loss}[1]{{\mathcal\left({#1}\right)}}
\newcommand{\conj}[1]{{\overline{#1}^{\top}}}
$$

In [ ]:
import deepinv as dinv
import torch

device = dinv.utils.get_device()

## Download the dataset

We download the live-cell mitochondria STED dataset from
[Zenodo](https://zenodo.org/records/14215838).
The dataset includes the low-intensity acquisitions (1.5 microwatt) as well as a reference high-intensity (8 microwatt) ground truth, which we use for visualisation.

In [ ]:
data_dir = dinv.utils.get_cache_home() / "datasets" / "sted"
archive_name = "live_cell_mitochondria_u2os_tom20_halotag7_dm_sir"

dinv.datasets.download_archive(
    f"https://zenodo.org/records/14215838/files/{archive_name}.rar?download=1",
    data_dir / f"{archive_name}.rar",
    extract=True,
)

## Build the dataset

We use [`deepinv.datasets.ImageFolder`](https://deepinv.org/api/stubs/deepinv.datasets.ImageFolder.html) to pair the ground truth and
low-intensity images. TIFF files are loaded with [`deepinv.utils.load_tiff`](https://deepinv.org/api/stubs/deepinv.utils.load_tiff.html).

In [ ]:
dataset = dinv.datasets.ImageFolder(
    data_dir / archive_name / "test_and_training_data_1",
    x_path="ground_truth_images/*.tif",
    y_path="low_intensity_images/*.tif",
    loader=lambda f: dinv.utils.load_tiff(f).squeeze(0).float(),
)

## Denoise with the RAM foundation model

[`deepinv.models.RAM`](https://deepinv.org/api/stubs/deepinv.models.RAM.html) is a pretrained foundation model for image
restoration. We process the data in batches. The `gain`
argument sets the Poisson noise level used by the model.

In [ ]:
model = dinv.models.RAM(device=device)

for x, y in torch.utils.data.DataLoader(
    dataset, batch_size=1
):  # adjust batch size as necessary
    x, y = x.to(device), y.to(device)
    with torch.no_grad():
        rescale = y.max()
        x_net = model(y / rescale, gain=0.15) * rescale

    break  # remove this to process the whole dataset

## Visualize the results

We compare the ground truth, the noisy low-intensity measurement, and the
denoised reconstruction.

In [ ]:
dinv.utils.plot(
    {
        "8 µW excitation": x,
        "1.5 µW excitation": y,
        "1.5 µW excitation\ndenoised": x_net,
    },
    figsize=(7, 5),
)

## Interactive viewer with napari

Use [`deepinv.utils.plot_napari`](https://deepinv.org/api/stubs/deepinv.utils.plot_napari.html) to interactively visualise microscopy images/stacks/volumes
with the [napari](https://napari.org) viewer.

Since it opens an interactive window, it requires a display and is not executed in this gallery.
To reproduce the screenshot below, run:

```python
dinv.utils.plot_napari(x, y, x_net, screenshot=False)
```
Pass ``screenshot=True`` to take a static screenshot instead:

<img src="https://huggingface.co/datasets/deepinv/images/resolve/main/demo_microscopy_denoising_screenshot.png" width="600" align="center" alt="napari viewer showing the ground truth, noisy and denoised images side by side">

## References

- Osuna-Vargas, Pamela and Wehrheim, Maren H. and Zinz, Lucas and Rahm, Johanna and Balakrishnan, Ashwin and Kaminer, Alexandra and Heilemann, Mike and Kaschube, Matthias (2025). *Denoising Diffusion Models for High-Resolution Microscopy Image Restoration*. Proceedings of the IEEE/CVF Winter Conference on Applications of Computer Vision (WACV).
